<a href="https://colab.research.google.com/github/mayar22mostafa/Dental-Cosmetics-AI/blob/main/Notebooks/stable_diffusion_2_inpainting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Local Inference on GPU
Model page: https://huggingface.co/sd2-community/stable-diffusion-2-inpainting

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/sd2-community/stable-diffusion-2-inpainting)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
import torch
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image, ImageFilter
import os

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "sd2-community/stable-diffusion-2-inpainting",
    torch_dtype=torch.float16,
)
pipe.to("cuda")   # change to "mps" for Apple Silicon, "cpu" if no GPU
pipe.enable_attention_slicing()   # save VRAM

In [ ]:
# ── Images (already resized to 512x512) ───────────────────────────────────────
image      = Image.open("/content/kkk.png").convert("RGB")
mask_image = Image.open("/content/kkkmask.png").convert("RGB")

# ── Prompts ───────────────────────────────────────────────────────────────────
prompt = (
    "perfectly straight, evenly spaced, well-aligned white teeth, "
    "after orthodontic braces treatment, healthy pink gums, "
    "natural beautiful smile, photorealistic, sharp focus, "
    "professional dental photography, 8k resolution"
)

negative_prompt = (
    "crooked teeth, misaligned, gaps, overcrowding, overlapping, "
    "yellow stains, broken, missing teeth, cartoon, painting, "
    "blurry, unrealistic, deformed, ugly, bad anatomy, "
    "text, watermark, logo"
)

# ── Generate (run 4 variations, pick the best) ────────────────────────────────
results = pipe(
    prompt              = prompt,
    negative_prompt     = negative_prompt,
    image               = image,
    mask_image          = mask_image,
    guidance_scale      = 12.0,   # 10-14 works best for dental
    num_inference_steps = 75,     # more steps = more detail
    strength            = 0.99,   # max: let SD fully redo the masked area
    num_images_per_prompt = 4,    # generate 4 options
).images

In [ ]:
# ── Save ──────────────────────────────────────────────────────────────────────
os.makedirs("outputs", exist_ok=True)
for i, img in enumerate(results):
    img.save(f"outputs/after_treatment_{i+1}.png")
    print(f"Saved: outputs/after_treatment_{i+1}.png")

print("\n✅ Done! Check the 'outputs' folder and pick the best result.")
print("💡 Tip: run multiple times — SD is random, some runs are much better than others.")